# 01 — Build and verify the matched dataset

This notebook pins the environment, checks HF/J-Lens agreement, constructs all 118 reciprocal prompt pairs, selects the WRITTEN layer and threshold using calibration groups only, and freezes the 77 verified held-out pairs. Calibration-only causal interchange is injected explicitly; no causal result is placed in the sanitized manifest consumed by cheap READ.

**Outputs:** `artifacts/final/01_dataset.json`, `01_clean_manifest.json`, and `01_directions.pt`.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
ARTIFACTS = ROOT / 'artifacts' / 'final'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

from src.causal_read import (
    clean_state_and_logits,
    symmetric_interchange,
    token_difference_metric,
)
from src.datasets import build_and_verify_dataset
from src.jlens_interface import (
    MODEL_ID,
    load_model,
    load_published_lens,
    release_model,
)

def write_json(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, sort_keys=True) + '\n')

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def progress(event: str, payload: dict) -> None:
    print(event, json.dumps(payload, sort_keys=True))

hf = shutil.which('hf')
if hf is None:
    raise RuntimeError('hf CLI is required')
preflight = {
    'hf_path': hf,
    'hf_identity': subprocess.run([hf, 'auth', 'whoami'], check=True, capture_output=True, text=True).stdout.strip(),
    'gpu_csv': subprocess.run(['nvidia-smi', '--query-gpu=memory.total,memory.free', '--format=csv'], check=True, capture_output=True, text=True).stdout.strip(),
    'torch_version': torch.__version__,
}
print(json.dumps(preflight, indent=2))

bundle = load_model(MODEL_ID)
published_lens = load_published_lens(MODEL_ID)
try:
    built = build_and_verify_dataset(
        bundle,
        published_lens,
        clean_state_and_logits=clean_state_and_logits,
        symmetric_interchange=symmetric_interchange,
        token_difference_metric=token_difference_metric,
        progress=progress,
    )
finally:
    del published_lens
    del bundle
    release_model()

direction_path = ARTIFACTS / '01_directions.pt'
torch.save(built['direction_cache'], direction_path)
direction_record = {
    'path': str(direction_path.relative_to(ROOT)),
    'bytes': direction_path.stat().st_size,
    'sha256': sha256(direction_path),
}
clean_manifest = built['sanitized_manifest']
clean_manifest['direction_cache'].update(direction_record)
clean_path = ARTIFACTS / '01_clean_manifest.json'
write_json(clean_path, clean_manifest)

dataset = built['full_dataset_artifact']
dataset['preflight'] = preflight
dataset['direction_cache'].update(direction_record)
dataset['clean_read_manifest'].update({
    'path': str(clean_path.relative_to(ROOT)),
    'bytes': clean_path.stat().st_size,
    'sha256': sha256(clean_path),
})
dataset_path = ARTIFACTS / '01_dataset.json'
write_json(dataset_path, dataset)

summary = {
    'selection': dataset['selection']['layer'],
    'written_threshold': dataset['selection']['written_threshold'],
    'counts': dataset['counts'],
    'kl_max_mean': dataset['logit_agreement']['max_mean_kl'],
    'dataset_sha256': sha256(dataset_path),
    'clean_manifest_sha256': sha256(clean_path),
    'direction_cache_sha256': sha256(direction_path),
}
print(json.dumps(summary, indent=2))
